# Final constellation submission pipeline

This notebook regenerates the **final candidate that received 0.94 on Kaggle** from the supplied validation sky images and query patches. It runs the equal-weight and raw-heavy matchers separately, then applies the conservative pixel-evidence fusion. The raw-heavy output is **not** the file to submit.

Run from the repository root with the project's `.venv` Python kernel. Install `requirements.txt` in that kernel if needed. The first uncached run can take hours on this device; later runs reuse the local `outputs/final/cache` evidence. Only trust a cache created locally by this code. No training labels, selected reference CSV, external star catalogue, or synthetic images enter validation inference.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import hashlib
import json
import subprocess
import sys

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'constellation' / 'robust_engine.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the constellation repository.')
DATA = ROOT / 'data' / 'constellation-detection' / 'participant'
if not (DATA / 'sample_submission.csv').is_file():
    raise FileNotFoundError(f'Missing competition files under {DATA}')
OUTPUT = ROOT / 'outputs' / 'final'
CACHE = OUTPUT / 'cache'
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('Dataset:', DATA)
print('Final CSV:', OUTPUT / 'submission.csv')
print('Trusted local cache:', CACHE)


## Regenerate predictions and fuse them

This cell can take a long time. Its intermediate CSVs live in a temporary directory and are removed after the validated final CSV is written; reusable image evidence remains in `outputs/final/cache`. If the cell fails, fix the error and rerun it.


In [ ]:
def run_module(module, *arguments):
    command = [sys.executable, '-m', module, *map(str, arguments)]
    print('Running:', module, flush=True)
    subprocess.run(command, cwd=ROOT, check=True)

with TemporaryDirectory(prefix='constellation-final-') as temporary:
    equal_dir = Path(temporary) / 'equal'
    raw_dir = Path(temporary) / 'raw'
    common = ('--data', DATA, '--robust-cache-dir', CACHE)
    run_module('constellation', 'robust-predict', *common, '--output', equal_dir)
    run_module('constellation', 'robust-predict', *common,
               '--robust-channel-weights', '1.5', '1', '0.5', '--output', raw_dir)
    run_module('constellation.conservative_fusion',
               '--data', DATA, '--split', 'validation',
               '--base', equal_dir / 'submission.csv',
               '--raw', raw_dir / 'submission.csv',
               '--cache', CACHE, '--output', OUTPUT)

FINAL_CSV = OUTPUT / 'submission.csv'
print('Final candidate:', FINAL_CSV)


## Verify the submission

The fusion command already validates legal constellation names, every patch tuple, image bounds, the exact columns and scene IDs. This last check also displays the file hash and the small set of changes made by the fusion. Submit only the final `outputs/final/submission.csv`.


In [ ]:
FINAL_CSV = OUTPUT / 'submission.csv'
with (DATA / 'sample_submission.csv').open(newline='', encoding='utf-8') as stream:
    template = csv.DictReader(stream)
    expected_fields = template.fieldnames
    expected_ids = [row['Id'] for row in template]
with FINAL_CSV.open(newline='', encoding='utf-8') as stream:
    submission = csv.DictReader(stream)
    actual_fields = submission.fieldnames
    rows = list(submission)
assert actual_fields == expected_fields
assert [row['Id'] for row in rows] == expected_ids
assert len(rows) == 16 and sum(int(row['n_patches']) for row in rows) == 668
report = json.loads((OUTPUT / 'fusion-report.json').read_text(encoding='utf-8'))
digest = hashlib.sha256(FINAL_CSV.read_bytes()).hexdigest()
print(f'Validated {len(rows)} scenes and 668 patches')
print('Fusion changes:', report['changes'])
print('SHA-256:', digest)
print('Submit:', FINAL_CSV.resolve())
